# Predict Fire PM2.5 for 17 CMIP6 models × 4 SSPs and save daily GeoTIFFs

This notebook loads `final_model.pt`, applies the original 41-channel
normalization, adds DOY and spatial channels, and writes one georeferenced
Fire PM2.5 GeoTIFF per available SSP predictor date.

Existing valid TIFFs are skipped, so inference can resume after interruption.


In [ ]:
import json
import math
import os
import time
import traceback
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from rasterio.windows import Window
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
# =============================================================================
# SSP INFERENCE CONFIGURATION
# =============================================================================
PROJECT_DIR = Path(os.environ.get("AU_FIRE_ROOT", str(Path.cwd().parent / "data")))
PREDICTOR_ROOT = (
    PROJECT_DIR / "tensor/ssp_predictor_weather_lag30_totemi_raw"
)
OUTPUT_ROOT = PROJECT_DIR / "prediction/firepm25_ssp_daily_tif"
CHECKPOINT_PATH = (
    PROJECT_DIR
    / "checkpoints/firepm25_rawflux_unet_final_2000-2023_cap200_beta25"
    / "final_model.pt"
)
STATS_PATH = (
    PROJECT_DIR / "tensor/predictor_weather_lag30_totemi_raw.pt"
)
REFERENCE_RASTER = PROJECT_DIR / "data/elevation.tif"
MASK_PATH = PROJECT_DIR / "data/aus_mask.feather"

MODELS = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "CMCC-ESM2",
    "EC-Earth3",
    "EC-Earth3-Veg-LR",
    "GFDL-ESM4",
    "INM-CM4-8",
    "INM-CM5-0",
    "KACE-1-0-G",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM",
    "TaiESM1",
]
SSPS = ["ssp126", "ssp245", "ssp370", "ssp585"]

# First test one combination, then use MODELS.copy() and SSPS.copy().
SELECTED_MODELS = [MODELS[0]]
SELECTED_SSPS = ["ssp126"]

PREDICTION_START = "2015-01-01"
PREDICTION_END = "2100-12-31"

BATCH_SIZE = 32
NUM_WORKERS = 8
PREFETCH_FACTOR = 1
DEVICE_IDS = [0]
RAW_PREDICTION_CAP = 1000.0

OVERWRITE_EXISTING = False
VERIFY_EXISTING_TIF = False
FAIL_ON_MISSING_PREDICTOR = True
NODATA_VALUE = -9999.0

AU_ROW_START = 396
AU_COL_START = 1107
AU_HEIGHT = 140
AU_WIDTH = 284

if torch.cuda.is_available() and DEVICE_IDS:
    device = torch.device(f"cuda:{DEVICE_IDS[0]}")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

assert all(isinstance(value, str) for value in SELECTED_MODELS)
assert all(isinstance(value, str) for value in SELECTED_SSPS)
assert set(SELECTED_MODELS).issubset(MODELS)
assert set(SELECTED_SSPS).issubset(SSPS)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
# =========================
# UNet 配置转换函数
# =========================
@dataclass
class UNetConfig:
    """UNet 模型配置"""
    in_channels: int = 50
    out_channels: int = 1
    base_channels: int = 64
    
    # Dropout 配置
    encoder_dropout: float = 0.0
    bottleneck_dropout: float = 0.4
    decoder_dropouts: tuple = (0.3, 0.2, 0.1, 0.0)
    
    # Dropout 选项
    use_dropout2d: bool = True
    dropout_position: str = 'between'
    
    # BatchNorm 配置
    use_batchnorm: bool = True
    batchnorm_momentum: float = 0.1
    
    # 激活函数
    activation: str = 'relu'
    
    def __post_init__(self):
        """验证配置参数"""
        assert len(self.decoder_dropouts) == 4, "decoder_dropouts 必须包含 4 个值"
        assert 0 <= self.encoder_dropout <= 1, "dropout 值必须在 [0, 1] 之间"
        assert 0 <= self.bottleneck_dropout <= 1, "dropout 值必须在 [0, 1] 之间"
        assert all(0 <= d <= 1 for d in self.decoder_dropouts), "dropout 值必须在 [0, 1] 之间"


def args_to_unet_config(args):
    """将 args 转换为 UNetConfig"""
    return UNetConfig(
        in_channels=args.input_channels,
        out_channels=args.output_channels,
        base_channels=args.base_channels,
        encoder_dropout=args.encoder_dropout,
        bottleneck_dropout=args.bottleneck_dropout,
        decoder_dropouts=(
            args.decoder_dropout_1,
            args.decoder_dropout_2,
            args.decoder_dropout_3,
            args.decoder_dropout_4
        ),
        use_dropout2d=args.use_dropout2d,
        dropout_position=args.dropout_position,
        use_batchnorm=args.use_batchnorm,
        batchnorm_momentum=args.batchnorm_momentum,
        activation=args.activation
    )


# =========================
# 基础模块
# =========================
class DoubleConv(nn.Module):
    """双卷积模块"""
    
    def __init__(self, in_channels, out_channels, config: UNetConfig, dropout=0.0):
        super().__init__()
        
        self.dropout = dropout
        self.config = config
        
        # 第一个卷积块
        conv1_layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        ]
        if config.use_batchnorm:
            conv1_layers.append(nn.BatchNorm2d(out_channels, momentum=config.batchnorm_momentum))
        conv1_layers.append(self._get_activation())
        self.conv1 = nn.Sequential(*conv1_layers)
        
        # Dropout（在两个 conv 之间）
        if dropout > 0 and config.dropout_position == 'between':
            if config.use_dropout2d:
                self.dropout_layer = nn.Dropout2d(p=dropout)
            else:
                self.dropout_layer = nn.Dropout(p=dropout)
        else:
            self.dropout_layer = None
        
        # 第二个卷积块
        conv2_layers = [
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        ]
        if config.use_batchnorm:
            conv2_layers.append(nn.BatchNorm2d(out_channels, momentum=config.batchnorm_momentum))
        conv2_layers.append(self._get_activation())
        self.conv2 = nn.Sequential(*conv2_layers)
        
        # Dropout（在两个 conv 之后）
        if dropout > 0 and config.dropout_position == 'after':
            if config.use_dropout2d:
                self.dropout_after = nn.Dropout2d(p=dropout)
            else:
                self.dropout_after = nn.Dropout(p=dropout)
        else:
            self.dropout_after = None

    def _get_activation(self):
        """根据配置返回激活函数"""
        if self.config.activation == 'relu':
            return nn.ReLU(inplace=True)
        elif self.config.activation == 'leaky_relu':
            return nn.LeakyReLU(0.2, inplace=True)
        elif self.config.activation == 'gelu':
            return nn.GELU()
        else:
            raise ValueError(f"不支持的激活函数: {self.config.activation}")

    def forward(self, x):
        x = self.conv1(x)
        if self.dropout_layer is not None:
            x = self.dropout_layer(x)
        x = self.conv2(x)
        if self.dropout_after is not None:
            x = self.dropout_after(x)
        return x


def pad_to_match(x, ref):
    """将 x 填充到与 ref 相同的尺寸"""
    diff_y = ref.size(2) - x.size(2)
    diff_x = ref.size(3) - x.size(3)
    return F.pad(x, [0, diff_x, 0, diff_y])


# =========================
# UNet 模型
# =========================
class UNet(nn.Module):
    """可配置的 UNet 模型"""
    
    def __init__(self, config: UNetConfig):
        super().__init__()
        self.config = config
        
        bc = config.base_channels
        
        # ==================== Encoder ====================
        self.enc1 = DoubleConv(config.in_channels, bc, config, dropout=config.encoder_dropout)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv(bc, bc * 2, config, dropout=config.encoder_dropout)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = DoubleConv(bc * 2, bc * 4, config, dropout=config.encoder_dropout)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = DoubleConv(bc * 4, bc * 8, config, dropout=config.encoder_dropout)
        self.pool4 = nn.MaxPool2d(2)
        
        # ==================== Bottleneck ====================
        self.bottleneck = DoubleConv(
            bc * 8, 
            bc * 16, 
            config, 
            dropout=config.bottleneck_dropout
        )
        
        # ==================== Decoder ====================
        self.up4 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec4 = DoubleConv(bc * 16 + bc * 8, bc * 8, config, dropout=config.decoder_dropouts[0])
        
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec3 = DoubleConv(bc * 8 + bc * 4, bc * 4, config, dropout=config.decoder_dropouts[1])
        
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec2 = DoubleConv(bc * 4 + bc * 2, bc * 2, config, dropout=config.decoder_dropouts[2])
        
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.dec1 = DoubleConv(bc * 2 + bc, bc, config, dropout=config.decoder_dropouts[3])
        
        # ==================== Output ====================
        self.out_conv = nn.Conv2d(bc, config.out_channels, kernel_size=1)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """权重初始化"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))
        
        # Decoder
        u4 = self.up4(b)
        u4 = pad_to_match(u4, e4)
        d4 = self.dec4(torch.cat([u4, e4], dim=1))
        
        u3 = self.up3(d4)
        u3 = pad_to_match(u3, e3)
        d3 = self.dec3(torch.cat([u3, e3], dim=1))
        
        u2 = self.up2(d3)
        u2 = pad_to_match(u2, e2)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        
        u1 = self.up1(d2)
        u1 = pad_to_match(u1, e1)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        
        return self.out_conv(d1)


# =========================
# 模型初始化函数
# =========================
def create_model(args, verbose=True):
    """
    从 args 创建 UNet 模型
    
    Args:
        args: 包含所有超参数的 argparse.Namespace
        verbose: 是否打印模型信息
    
    Returns:
        model: UNet 模型
        config: UNetConfig 配置对象
    """
    # 转换配置
    config = args_to_unet_config(args)
    
    # 创建模型
    model = UNet(config)
    
    if verbose:
        print("=" * 70)
        print("🔥 UNet Model Configuration")
        print("=" * 70)
        print(f"📊 Architecture:")
        print(f"  - Input channels:     {config.in_channels}")
        print(f"  - Output channels:    {config.out_channels}")
        print(f"  - Base channels:      {config.base_channels}")
        print(f"\n🛡️  Dropout Configuration:")
        print(f"  - Encoder dropout:    {config.encoder_dropout}")
        print(f"  - Bottleneck dropout: {config.bottleneck_dropout}")
        print(f"  - Decoder dropouts:   {config.decoder_dropouts}")
        print(f"  - Dropout type:       {'Dropout2d' if config.use_dropout2d else 'Dropout'}")
        print(f"  - Dropout position:   {config.dropout_position}")
        print(f"\n⚙️  Other Settings:")
        print(f"  - BatchNorm:          {config.use_batchnorm}")
        print(f"  - Activation:         {config.activation}")
        print(f"  - Weight decay:       {args.weight_decay}")
        
        # 参数统计
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\n📈 Parameters:")
        print(f"  - Total:              {total_params:,}")
        print(f"  - Trainable:          {trainable_params:,}")
        print("=" * 70)
    
    # 多 GPU 设置
    if len(args.device_ids) > 1:
        model = nn.DataParallel(model, device_ids=args.device_ids)
        if verbose:
            print(f"✅ Using DataParallel with GPUs: {args.device_ids}")
    
    # 移动到设备
    device = torch.device(f"cuda:{args.device_ids[0]}" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    return model, config


# =========================
# 预设配置函数
# =========================
def get_light_dropout_args(base_args):
    """轻度 dropout 配置（适合轻微过拟合）"""
    args = argparse.Namespace(**vars(base_args))
    args.encoder_dropout = 0.0
    args.bottleneck_dropout = 0.2
    args.decoder_dropout_1 = 0.15
    args.decoder_dropout_2 = 0.1
    args.decoder_dropout_3 = 0.05
    args.decoder_dropout_4 = 0.0
    args.experiment_name = "firepm25_light_dropout"
    return args


def get_medium_dropout_args(base_args):
    """中度 dropout 配置（推荐默认配置）"""
    args = argparse.Namespace(**vars(base_args))
    args.encoder_dropout = 0.0
    args.bottleneck_dropout = 0.4
    args.decoder_dropout_1 = 0.3
    args.decoder_dropout_2 = 0.2
    args.decoder_dropout_3 = 0.1
    args.decoder_dropout_4 = 0.0
    args.experiment_name = "firepm25_medium_dropout"
    return args


def get_aggressive_dropout_args(base_args):
    """激进 dropout 配置（适合严重过拟合）"""
    args = argparse.Namespace(**vars(base_args))
    args.encoder_dropout = 0.1
    args.bottleneck_dropout = 0.5
    args.decoder_dropout_1 = 0.4
    args.decoder_dropout_2 = 0.3
    args.decoder_dropout_3 = 0.2
    args.decoder_dropout_4 = 0.1
    args.experiment_name = "firepm25_aggressive_dropout"
    return args


def get_no_dropout_args(base_args):
    """无 dropout 配置（baseline）"""
    args = argparse.Namespace(**vars(base_args))
    args.encoder_dropout = 0.0
    args.bottleneck_dropout = 0.0
    args.decoder_dropout_1 = 0.0
    args.decoder_dropout_2 = 0.0
    args.decoder_dropout_3 = 0.0
    args.decoder_dropout_4 = 0.0
    args.experiment_name = "firepm25_no_dropout"
    return args


# =========================
# 使用示例
# =========================
# if __name__ == "__main__":
#     # 方式 1: 使用默认配置（中度 dropout）
#     print("\n🔹 方式 1: 默认配置")
#     model, config = create_model(args)
    
#     # 方式 2: 使用预设配置
#     print("\n🔹 方式 2: 激进 dropout 配置")
#     aggressive_args = get_aggressive_dropout_args(args)
#     model_aggressive, _ = create_model(aggressive_args)
    
#     # 方式 3: 自定义 dropout 值
#     print("\n🔹 方式 3: 自定义配置")
#     custom_args = argparse.Namespace(**vars(args))
#     custom_args.encoder_dropout = 0.0
#     custom_args.bottleneck_dropout = 0.15
#     custom_args.decoder_dropout_1 = 0.10
#     custom_args.decoder_dropout_2 = 0.05
#     custom_args.decoder_dropout_3 = 0.0
#     custom_args.decoder_dropout_4 = 0.0
#     custom_args.experiment_name = "firepm25_custom_dropout"
#     model_custom, _ = create_model(custom_args)
    
#     # 测试前向传播
#     print("\n🔹 测试前向传播")
#     device = torch.device(f"cuda:{args.device_ids[0]}" if torch.cuda.is_available() else "cpu")
#     x = torch.randn(2, 50, 256, 256).to(device)
    
#     with torch.no_grad():
#         y = model(x)
    
#     print(f"Input shape:  {x.shape}")
#     print(f"Output shape: {y.shape}")
#     print("\n✅ 模型测试通过！")

# # --- Instantiate model ---
# model = UNet(in_channels=args.input_channels, base_channels=args.base_channels)

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
config = checkpoint.get("config", {})
required_config = {
    "input_channels": 45,
    "output_channels": 1,
    "base_channels": 64,
}
for key, expected in required_config.items():
    if config.get(key) != expected:
        raise ValueError(
            f"Checkpoint {key}={config.get(key)}; expected {expected}"
        )

model_args = type("ModelArgs", (), config)()
model = UNet(args_to_unet_config(model_args))
state_dict = checkpoint.get("model", checkpoint)
state_dict = {
    (key[7:] if key.startswith("module.") else key): value
    for key, value in state_dict.items()
}
model.load_state_dict(state_dict)
model = model.to(device).eval()
print(
    f"Loaded final model trained on "
    f"{checkpoint.get('training_start')} to {checkpoint.get('training_end')}"
)

climate_mean, climate_std = torch.load(
    STATS_PATH,
    map_location="cpu",
)
climate_mean = torch.as_tensor(climate_mean, dtype=torch.float32).flatten()
climate_std = torch.as_tensor(climate_std, dtype=torch.float32).flatten()
if climate_mean.numel() != 41 or climate_std.numel() != 41:
    raise ValueError("Expected 41-channel normalization statistics")

mask_array = pd.read_feather(MASK_PATH).to_numpy()
if mask_array.shape != (AU_HEIGHT, AU_WIDTH):
    mask_array = mask_array.reshape(AU_HEIGHT, AU_WIDTH)
land_mask = np.isfinite(mask_array) & (mask_array != 0)

with rasterio.open(REFERENCE_RASTER) as reference:
    if reference.height == AU_HEIGHT and reference.width == AU_WIDTH:
        output_transform = reference.transform
    else:
        output_transform = reference.window_transform(
            Window(
                col_off=AU_COL_START,
                row_off=AU_ROW_START,
                width=AU_WIDTH,
                height=AU_HEIGHT,
            )
        )
    output_crs = reference.crs

if output_crs is None:
    raise ValueError("Reference raster has no CRS")


In [ ]:
def date_strings(start_date, end_date):
    return [
        f"{date:%Y-%m-%d}"
        for date in pd.date_range(start_date, end_date, freq="D")
    ]


class SSPPredictorDataset(Dataset):
    def __init__(self, predictor_dir, dates):
        self.predictor_dir = Path(predictor_dir)
        self.dates = list(dates)
        lat_norm = torch.linspace(1.0, -1.0, AU_HEIGHT)
        lon_norm = torch.linspace(-1.0, 1.0, AU_WIDTH)
        latitude = lat_norm[:, None].expand(
            AU_HEIGHT,
            AU_WIDTH,
        ).unsqueeze(0)
        longitude = lon_norm[None, :].expand(
            AU_HEIGHT,
            AU_WIDTH,
        ).unsqueeze(0)
        self.spatial = torch.cat([latitude, longitude], dim=0)

    def __len__(self):
        return len(self.dates)

    def __getitem__(self, index):
        date = self.dates[index]
        path = self.predictor_dir / f"{date}.pt"
        x = torch.load(path, map_location="cpu").float()
        if x.shape != (41, AU_HEIGHT, AU_WIDTH):
            raise ValueError(f"Unexpected predictor shape {x.shape}: {path}")
        x = (
            x - climate_mean[:, None, None]
        ) / (climate_std[:, None, None] + 1e-6)
        x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        parsed = datetime.strptime(date, "%Y-%m-%d")
        day_of_year = parsed.timetuple().tm_yday
        period = 365.2425
        sin_doy = torch.full(
            (1, AU_HEIGHT, AU_WIDTH),
            math.sin(2 * math.pi * day_of_year / period),
        )
        cos_doy = torch.full(
            (1, AU_HEIGHT, AU_WIDTH),
            math.cos(2 * math.pi * day_of_year / period),
        )
        x = torch.cat(
            [x, sin_doy, cos_doy, self.spatial],
            dim=0,
        )
        return x, date


def make_loader(dataset):
    kwargs = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
        kwargs["persistent_workers"] = True
    return DataLoader(**kwargs)


In [ ]:
def tif_is_valid(path):
    if not path.is_file():
        return False
    if not VERIFY_EXISTING_TIF:
        return True
    try:
        with rasterio.open(path) as dataset:
            return (
                dataset.count == 1
                and dataset.height == AU_HEIGHT
                and dataset.width == AU_WIDTH
            )
    except Exception:
        return False


def write_prediction_tif_atomic(array, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(".tif.tmp")
    output = np.asarray(array, dtype=np.float32)
    output = np.where(
        land_mask & np.isfinite(output),
        output,
        NODATA_VALUE,
    ).astype(np.float32)

    profile = {
        "driver": "GTiff",
        "height": AU_HEIGHT,
        "width": AU_WIDTH,
        "count": 1,
        "dtype": "float32",
        "crs": output_crs,
        "transform": output_transform,
        "nodata": NODATA_VALUE,
        "compress": "deflate",
        "predictor": 3,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 128,
        "BIGTIFF": "IF_SAFER",
    }
    with rasterio.open(temporary_path, "w", **profile) as destination:
        destination.write(output, 1)
    os.replace(temporary_path, output_path)


@torch.no_grad()
def predict_one_combination(model_name, ssp_name):
    predictor_dir = PREDICTOR_ROOT / ssp_name / model_name
    output_dir = OUTPUT_ROOT / ssp_name / model_name
    requested_dates = date_strings(PREDICTION_START, PREDICTION_END)

    missing_dates = [
        date
        for date in requested_dates
        if not (predictor_dir / f"{date}.pt").is_file()
    ]
    output_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"date": missing_dates}).to_csv(
        output_dir / "missing_predictor_dates.csv",
        index=False,
    )
    if missing_dates and FAIL_ON_MISSING_PREDICTOR:
        raise FileNotFoundError(
            f"{model_name}/{ssp_name}: {len(missing_dates)} predictor dates missing; "
            f"first={missing_dates[0]}"
        )

    missing_date_set = set(missing_dates)
    available_dates = [
        date
        for date in requested_dates
        if date not in missing_date_set
    ]
    pending_dates = [
        date
        for date in available_dates
        if OVERWRITE_EXISTING
        or not tif_is_valid(output_dir / f"{date}.tif")
    ]
    if not pending_dates:
        return {
            "model": model_name,
            "ssp": ssp_name,
            "requested": len(requested_dates),
            "predicted": 0,
            "skipped_existing": len(available_dates),
            "missing_predictor": len(missing_dates),
        }

    dataset = SSPPredictorDataset(predictor_dir, pending_dates)
    loader = make_loader(dataset)
    daily_rows = []
    for x, dates in tqdm(
        loader,
        desc=f"{model_name} {ssp_name}",
        leave=False,
    ):
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda"),
        ):
            prediction = F.softplus(model(x).float()).clamp(
                max=RAW_PREDICTION_CAP
            )
        prediction = prediction.squeeze(1).cpu().numpy()
        for index, date in enumerate(dates):
            array = prediction[index]
            write_prediction_tif_atomic(
                array,
                output_dir / f"{date}.tif",
            )
            valid_values = array[land_mask & np.isfinite(array)]
            daily_rows.append(
                {
                    "date": date,
                    "mean": float(valid_values.mean()),
                    "std": float(valid_values.std()),
                    "p95": float(np.quantile(valid_values, 0.95)),
                    "p99": float(np.quantile(valid_values, 0.99)),
                    "max": float(valid_values.max()),
                }
            )

    if daily_rows:
        new_daily = pd.DataFrame(daily_rows)
        summary_path = output_dir / "daily_prediction_summary.csv"
        if summary_path.is_file() and not OVERWRITE_EXISTING:
            old_daily = pd.read_csv(summary_path)
            new_daily = (
                pd.concat([old_daily, new_daily], ignore_index=True)
                .drop_duplicates("date", keep="last")
                .sort_values("date")
            )
        new_daily.to_csv(summary_path, index=False)

    return {
        "model": model_name,
        "ssp": ssp_name,
        "requested": len(requested_dates),
        "predicted": len(pending_dates),
        "skipped_existing": len(available_dates) - len(pending_dates),
        "missing_predictor": len(missing_dates),
    }


In [ ]:
results = []
for model_name in SELECTED_MODELS:
    for ssp_name in SELECTED_SSPS:
        try:
            result = predict_one_combination(model_name, ssp_name)
            result["status"] = "OK"
        except Exception as error:
            result = {
                "model": model_name,
                "ssp": ssp_name,
                "status": "ERROR",
                "error": repr(error),
                "traceback": traceback.format_exc(),
            }
        results.append(result)
        print(result)

status = pd.DataFrame(results)
status.to_csv(OUTPUT_ROOT / "ssp_prediction_status.csv", index=False)
display(status)

failures = status.loc[status["status"] != "OK"]
if not failures.empty:
    raise RuntimeError(
        f"{len(failures)} SSP inference task(s) failed; "
        "see ssp_prediction_status.csv"
    )


## Output structure

`prediction/firepm25_ssp_daily_tif/{ssp}/{model}/{YYYY-MM-DD}.tif`

Every GeoTIFF is float32, uses the Australia-grid CRS/transform from
`data/elevation.tif`, has `-9999` nodata outside the land mask, and is
DEFLATE-compressed. Existing valid files are skipped when
`OVERWRITE_EXISTING=False`.
